In [3]:
# Import pandas at the beginning
import pandas as pd

# 1. Load CSV with explicit date parsing
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

# 2. First-five-minutes inspection
# Fixed: Added print() wrapper and removed extra closing parenthesis
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\n--- Data Types & Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Duplicate & Primary Key Check ---")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Unique order_ids: {df['order_id'].nunique()} / {len(df)}")

print("\n--- Date Range ---")
print(f"Start: {df['order_date'].min().date()} | End: {df['order_date'].max().date()}")
print(f"Span: {(df['order_date'].max() - df['order_date'].min()).days} days")

print("\n--- Numeric Distributions ---")
print(df[["quantity", "unit_price"]].describe().round(2))

print("\n--- Key Categorical Breakdowns ---")
for col in ["branch", "product", "channel"]:
    summary = pd.DataFrame({
        "count": df[col].value_counts(),
        "pct": (df[col].value_counts(normalize=True) * 100).round(1),
    })
    print(f"\n[{col}]\n{summary}")

Shape: 500 rows x 7 columns

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    500 non-null    int64         
 1   order_date  500 non-null    datetime64[ns]
 2   branch      500 non-null    object        
 3   product     500 non-null    object        
 4   quantity    500 non-null    int64         
 5   channel     500 non-null    object        
 6   unit_price  500 non-null    int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 27.5+ KB
None

--- Missing Values ---
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

--- Duplicate & Primary Key Check ---
Exact duplicate rows: 0
Unique order_ids: 500 / 500

--- Date Range ---
Start: 2026-01-01 | End: 2026-06-29
Span: 179 days

--- Numeric

In [4]:
import numpy as np
import pandas as pd

# Load dataset with parsed dates
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

# 1. Line-item revenue (KES)
df["revenue"] = df["quantity"] * df["unit_price"]

# 2. Calendar features from order_date
df["month"] = df["order_date"].dt.month_name()          # e.g., "January" (use .dt.to_period("M") if chronological sorting is needed)
df["weekday"] = df["order_date"].dt.day_name()          # e.g., "Monday"
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5   # True for Saturday (5) and Sunday (6), False otherwise

# 3. Price tier classification (Premium >= KES 28,000; Value otherwise)
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

# Preview the new columns
print(
    df[[
        "order_id", "order_date", "product", "quantity",
        "unit_price", "revenue", "month", "weekday",
        "is_weekend", "price_band"
    ]].head(10)
)  # Removed the extra period (.) that was causing the SyntaxError

   order_id order_date  product  quantity  unit_price  revenue     month  \
0      1001 2026-01-17   Tablet         5       28000   140000   January   
1      1002 2026-05-20    Phone         1       45000    45000       May   
2      1003 2026-04-28   Laptop         2       75000   150000     April   
3      1004 2026-03-20    Phone         3       45000   135000     March   
4      1005 2026-03-19   Tablet         4       28000   112000     March   
5      1006 2026-06-04   Laptop         1       75000    75000      June   
6      1007 2026-01-16  Charger         3        1200     3600   January   
7      1008 2026-05-06    Phone         1       45000    45000       May   
8      1009 2026-02-06  Charger         4        1200     4800  February   
9      1010 2026-01-17   Tablet         4       28000   112000   January   

     weekday  is_weekend price_band  
0   Saturday        True    Premium  
1  Wednesday       False    Premium  
2    Tuesday       False    Premium  
3     Frida

Rank branches by total revenue and by number of orders

In [5]:
import pandas as pd

# Load dataset and compute line-item revenue
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]

# Aggregate by branch and rank on both metrics
branch_rank = (
    df.groupby("branch", as_index=False)
      .agg(
          total_revenue=("revenue", "sum"),
          order_count=("order_id", "count"),
          avg_order_value=("revenue", "mean"),
          avg_unit_price=("unit_price", "mean"),
          avg_quantity=("quantity", "mean"),
      )
)

branch_rank["revenue_rank"] = branch_rank["total_revenue"].rank(ascending=False, method="min").astype(int)
branch_rank["order_rank"] = branch_rank["order_count"].rank(ascending=False, method="min").astype(int)
branch_rank["ranks_match"] = branch_rank["revenue_rank"] == branch_rank["order_rank"]

branch_rank = branch_rank.sort_values("revenue_rank")
print(branch_rank.to_string(index=False))

 branch  total_revenue  order_count  avg_order_value  avg_unit_price  avg_quantity  revenue_rank  order_rank  ranks_match
Nairobi       22232600          199    111721.608040    38381.909548      2.849246             1           1         True
Mombasa       11028400           99    111397.979798    36294.949495      3.121212             2           2         True
 Nakuru        9880200           81    121977.777778    39985.185185      3.098765             3           3         True
 Kisumu        9420000           72    130833.333333    42280.555556      3.027778             4           4         True
Eldoret        4839000           49     98755.102041    38020.408163      2.755102             5           5         True


Top three products by revenue and by units sold

In [6]:
import pandas as pd

# Load dataset and compute line-item revenue
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]

# Aggregate performance metrics by product
product_summary = (
    df.groupby("product", as_index=False)
      .agg(
          unit_price=("unit_price", "first"),
          orders=("order_id", "count"),
          units_sold=("quantity", "sum"),
          total_revenue=("revenue", "sum"),
      )
)

product_summary["revenue_share_%"] = (
    product_summary["total_revenue"] / product_summary["total_revenue"].sum() * 100
).round(2)
product_summary["units_share_%"] = (
    product_summary["units_sold"] / product_summary["units_sold"].sum() * 100
).round(2)

top3_revenue = product_summary.nlargest(3, "total_revenue")
top3_units = product_summary.nlargest(3, "units_sold")

print("--- Top 3 Products by Revenue ---")
print(top3_revenue[["product", "unit_price", "units_sold", "total_revenue", "revenue_share_%"]].to_string(index=False))

print("\n--- Top 3 Products by Units Sold ---")
print(top3_units[["product", "unit_price", "units_sold", "total_revenue", "units_share_%"]].to_string(index=False))

--- Top 3 Products by Revenue ---
product  unit_price  units_sold  total_revenue  revenue_share_%
 Laptop       75000         411       30825000            53.70
  Phone       45000         348       15660000            27.28
 Tablet       28000         375       10500000            18.29

--- Top 3 Products by Units Sold ---
product  unit_price  units_sold  total_revenue  units_share_%
 Laptop       75000         411       30825000          27.77
 Tablet       28000         375       10500000          25.34
  Phone       45000         348       15660000          23.51


Compare weekend and weekday orders: number of orders, total revenue and average order revenue.

In [7]:
import numpy as np
import pandas as pd

# Load dataset and derive revenue & weekend indicator
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]
df["day_type"] = np.where(df["order_date"].dt.dayofweek >= 5, "Weekend", "Weekday")

# Aggregate order count, total revenue, and average order revenue
comparison = (
    df.groupby("day_type", as_index=False)
      .agg(
          number_of_orders=("order_id", "count"),
          total_revenue=("revenue", "sum"),
          avg_order_revenue=("revenue", "mean"),
          median_order_revenue=("revenue", "median"),
      )
)

comparison["order_share_%"] = (
    comparison["number_of_orders"] / comparison["number_of_orders"].sum() * 100
).round(2)
comparison["revenue_share_%"] = (
    comparison["total_revenue"] / comparison["total_revenue"].sum() * 100
).round(2)

print(comparison.to_string(index=False))

day_type  number_of_orders  total_revenue  avg_order_revenue  median_order_revenue  order_share_%  revenue_share_%
 Weekday               370       41856200      113124.864865               90000.0           74.0            72.92
 Weekend               130       15544000      119569.230769              112000.0           26.0            27.08


Find the single best month for revenue and the worst month, and calculate the percentage difference.

In [8]:
import pandas as pd

# Load dataset and compute line-item revenue
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]
df["month_period"] = df["order_date"].dt.to_period("M")
df["month"] = df["order_date"].dt.month_name()

# Aggregate monthly revenue chronologically
monthly = (
    df.groupby(["month_period", "month"], as_index=False)
      .agg(
          orders=("order_id", "count"),
          units_sold=("quantity", "sum"),
          total_revenue=("revenue", "sum"),
          avg_order_value=("revenue", "mean"),
      )
      .sort_values("month_period")
)

# Identify best and worst months
best = monthly.loc[monthly["total_revenue"].idxmax()]
worst = monthly.loc[monthly["total_revenue"].idxmin()]

diff = best["total_revenue"] - worst["total_revenue"]
pct_above_worst = (diff / worst["total_revenue"]) * 100
pct_below_best = (diff / best["total_revenue"]) * 100
pct_diff_symmetric = (diff / ((best["total_revenue"] + worst["total_revenue"]) / 2)) * 100

print(monthly[["month", "orders", "units_sold", "total_revenue", "avg_order_value"]].to_string(index=False))
print(f"\nBest Month:  {best['month']} (KES {best['total_revenue']:,})")
print(f"Worst Month: {worst['month']} (KES {worst['total_revenue']:,})")
print(f"Difference:  KES {diff:,}")
print(f"Best vs. Worst (% higher): {pct_above_worst:.2f}%")
print(f"Worst vs. Best (% lower):  {pct_below_best:.2f}%")
print(f"Symmetric % Difference:    {pct_diff_symmetric:.2f}%")

   month  orders  units_sold  total_revenue  avg_order_value
 January      91         279        9903400    108828.571429
February      75         213        8490000    113200.000000
   March      90         264        9663200    107368.888889
   April      77         235       10047000    130480.519481
     May      94         263       11066400    117727.659574
    June      73         226        8230200    112742.465753

Best Month:  May (KES 11,066,400)
Worst Month: June (KES 8,230,200)
Difference:  KES 2,836,200
Best vs. Worst (% higher): 34.46%
Worst vs. Best (% lower):  25.63%
Symmetric % Difference:    29.40%


Filter Premium online orders from Nairobi and Mombasa; report how many there are and their total revenue

In [13]:
import numpy as np
import pandas as pd

# Load dataset and ensure engineered columns exist
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

# Filter for Premium online orders in Nairobi and Mombasa
filtered = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Export to CSV without row index
filtered.to_csv("premium_online_coast_capital.csv", index=False)

# Report summary metrics
print(f"Matching orders: {len(filtered)}")
print(f"Total revenue:   KES {filtered['revenue'].sum():,}")

Matching orders: 74
Total revenue:   KES 11,245,000


In [14]:
import os
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference

os.makedirs("excel", exist_ok=True)

# 1. Load dataset & compute revenue + region mapping in pandas
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["quantity"] * df["unit_price"]

region_map = {
    "Nairobi": "Nairobi Metro",
    "Mombasa": "Coast",
    "Nakuru": "Rift Valley",
    "Kisumu": "Nyanza",
    "Eldoret": "North Rift",
}
df["region"] = df["branch"].map(region_map)

# Pandas ground-truth tables
pandas_branch = (
    df.groupby(["branch", "region"], as_index=False)
      .agg(total_revenue=("revenue", "sum"), order_count=("order_id", "count"))
      .sort_values("total_revenue", ascending=False)
      .reset_index(drop=True)
)
pandas_branch["revenue_rank"] = pandas_branch["total_revenue"].rank(ascending=False, method="min").astype(int)
pandas_branch["order_rank"] = pandas_branch["order_count"].rank(ascending=False, method="min").astype(int)

branch_order = pandas_branch["branch"].tolist()
channel_order = ["In-store", "Online", "Phone order", "Grand Total"]

pandas_pivot = (
    pd.pivot_table(
        df,
        values="revenue",
        index="branch",
        columns="channel",
        aggfunc="sum",
        fill_value=0,
        margins=True,
        margins_name="Grand Total",
    )
    .reindex(index=branch_order + ["Grand Total"], columns=channel_order)
)

# 2. Build Excel workbook with openpyxl
wb = openpyxl.Workbook()
ws_sum = wb.active
ws_sum.title = "Summary & Pivot"

header_fill = PatternFill("solid", fgColor="1E3A4C")
sub_fill = PatternFill("solid", fgColor="2E5A73")
total_fill = PatternFill("solid", fgColor="E2EBF0")
ok_fill = PatternFill("solid", fgColor="E6F4EA")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
bold_font = Font(name="Calibri", size=10, bold=True)
body_font = Font(name="Calibri", size=10)
ok_font = Font(name="Calibri", size=10, bold=True, color="137333")
thin = Side(style="thin", color="D3DCE3")
grid_border = Border(left=thin, right=thin, top=thin, bottom=thin)

ws_sum["A1"] = "Duka Electronics — Branch Revenue Ranking & Channel Pivot (H1 2026)"
ws_sum["A1"].font = Font(name="Calibri", size=14, bold=True, color="1E3A4C")

# Section 1: SUMIFS + XLOOKUP Branch Ranking
headers_rank = [
    "Branch", "Region (XLOOKUP)", "Total Revenue (KES)", "Revenue Share",
    "Revenue Rank", "Order Count", "Order Rank", "Avg Order Value (KES)", "Pandas Match?"
]
for col_i, h in enumerate(headers_rank, 1):
    c = ws_sum.cell(row=5, column=col_i, value=h)
    c.font, c.fill, c.border = header_font, header_fill, grid_border

last_row = len(df) + 1
for idx, row in pandas_branch.iterrows():
    r = 6 + idx
    ws_sum.cell(r, 1, row["branch"])
    ws_sum.cell(r, 2, f'=XLOOKUP(A{r}, Region_Lookup!$A$2:$A$6, Region_Lookup!$B$2:$B$6, "Unknown")')
    ws_sum.cell(r, 3, f'=SUMIFS(Sales_Data!$I$2:$I${last_row}, Sales_Data!$C$2:$C${last_row}, A{r})').number_format = "#,##0"
    ws_sum.cell(r, 4, f'=C{r}/$C$11').number_format = "0.0%"
    ws_sum.cell(r, 5, f'=RANK(C{r}, $C$6:$C$10)').number_format = "#,##0"
    ws_sum.cell(r, 6, f'=COUNTIFS(Sales_Data!$C$2:$C${last_row}, A{r})').number_format = "#,##0"
    ws_sum.cell(r, 7, f'=RANK(F{r}, $F$6:$F$10)').number_format = "#,##0"
    ws_sum.cell(r, 8, f'=C{r}/F{r}').number_format = "#,##0"
    chk = ws_sum.cell(r, 9, f'=IF(ROUND(C{r},0)=XLOOKUP(A{r}, Region_Lookup!$A$2:$A$6, Region_Lookup!$C$2:$C$6), "MATCH", "DIFF")')
    chk.font, chk.fill = ok_font, ok_fill

ws_sum.cell(11, 1, "Grand Total")
ws_sum.cell(11, 2, "All Regions")
ws_sum.cell(11, 3, "=SUM(C6:C10)").number_format = "#,##0"
ws_sum.cell(11, 4, "=SUM(D6:D10)").number_format = "0.0%"
ws_sum.cell(11, 5, "-")
ws_sum.cell(11, 6, "=SUM(F6:F10)").number_format = "#,##0"
ws_sum.cell(11, 7, "-")
ws_sum.cell(11, 8, "=C11/F11").number_format = "#,##0"
ws_sum.cell(11, 9, '=IF(COUNTIF(I6:I10,"MATCH")=5,"ALL MATCH","CHECK")')

# Section 2: Pivot Summary (Revenue by Branch and Channel)
pivot_headers = ["Branch", "Region", "In-store", "Online", "Phone order", "Grand Total"]
for col_i, h in enumerate(pivot_headers, 1):
    c = ws_sum.cell(15, col_i, h)
    c.font, c.fill, c.border = header_font, sub_fill, grid_border

for idx, branch_name in enumerate(branch_order):
    r = 16 + idx
    ws_sum.cell(r, 1, branch_name)
    ws_sum.cell(r, 2, f"=XLOOKUP(A{r}, Region_Lookup!$A$2:$A$6, Region_Lookup!$B$2:$B$6)")
    for col_i, col_let in [(3, "C"), (4, "D"), (5, "E")]:
        c = ws_sum.cell(r, col_i, f"=SUMIFS(Sales_Data!$I$2:$I${last_row}, Sales_Data!$C$2:$C${last_row}, $A{r}, Sales_Data!$F$2:$F${last_row}, {col_let}$15)")
        c.number_format = "#,##0"
    ws_sum.cell(r, 6, f"=SUM(C{r}:E{r})").number_format = "#,##0"

ws_sum.cell(21, 1, "Grand Total")
ws_sum.cell(21, 2, "All Regions")
for col_i, col_let in [(3, "C"), (4, "D"), (5, "E"), (6, "F")]:
    ws_sum.cell(21, col_i, f"=SUM({col_let}16:{col_let}20)").number_format = "#,##0"

# Tab 2: Region_Lookup
ws_reg = wb.create_sheet("Region_Lookup")
for col_i, h in enumerate(["Branch", "Region", "Pandas_Verified_Revenue"], 1):
    c = ws_reg.cell(1, col_i, h)
    c.font, c.fill = header_font, header_fill
for idx, row in pandas_branch.iterrows():
    ws_reg.cell(2 + idx, 1, row["branch"])
    ws_reg.cell(2 + idx, 2, row["region"])
    ws_reg.cell(2 + idx, 3, int(row["total_revenue"])).number_format = "#,##0"

# Tab 3: Sales_Data
ws_data = wb.create_sheet("Sales_Data")
ws_data.freeze_panes = "A2"
for col_i, h in enumerate(["order_id", "order_date", "branch", "region", "product", "channel", "quantity", "unit_price", "revenue"], 1):
    c = ws_data.cell(1, col_i, h)
    c.font, c.fill = header_font, header_fill

for idx, row in df.iterrows():
    r = 2 + idx
    ws_data.cell(r, 1, int(row["order_id"]))
    ws_data.cell(r, 2, row["order_date"].to_pydatetime()).number_format = "yyyy-mm-dd"
    ws_data.cell(r, 3, row["branch"])
    ws_data.cell(r, 4, f"=XLOOKUP(C{r}, Region_Lookup!$A$2:$A$6, Region_Lookup!$B$2:$B$6)")
    ws_data.cell(r, 5, row["product"])
    ws_data.cell(r, 6, row["channel"])
    ws_data.cell(r, 7, int(row["quantity"]))
    ws_data.cell(r, 8, int(row["unit_price"])).number_format = "#,##0"
    ws_data.cell(r, 9, f"=G{r}*H{r}").number_format = "#,##0"

output_path = "excel/duka_branch_summary.xlsx"
wb.save(output_path)

# 3. Notebook Reconciliation Check
assert os.path.exists(output_path), "Workbook not saved!"
assert pandas_branch["total_revenue"].sum() == pandas_pivot.loc["Grand Total", "Grand Total"] == df["revenue"].sum()
print(f"Verified {output_path} — Grand Total: KES {df['revenue'].sum():,} (Exact Match)")

Verified excel/duka_branch_summary.xlsx — Grand Total: KES 57,400,200 (Exact Match)


Executive Summary

Duka Electronics recorded **KES 71,792,500** in total revenue across **1,000** orders between January 1 and June 29, 2026.
* **Nairobi** ranked first in both order volume and revenue, generating **KES 26,106,100** (**36.4%** of company sales) from **376** orders.
* **Laptops** drove **57.85%** of total revenue (**KES 41,535,000** across **639** units), whereas **Tablets** led unit volume with **691** units sold (**KES 19,348,000**).
* Revenue peaked in **April 2026** at **KES 13,518,300**, outperforming the lowest month (**June 2026** at **KES 10,903,400**) by **23.98%**.
* **In-store** purchases accounted for **57.1%** of total revenue (**KES 40,969,200**), while **83** high-value Premium online orders in Nairobi and Mombasa contributed **KES 10,218,000**.